# Tutorial 02 — Data Pipeline

Covers:
1. `OASFilter` — quality-filter antibody sequences
2. `CDRSpanExtractor` — IMGT CDR boundary extraction
3. `EmbeddingCache` — save / load per-residue structure embeddings
4. `AntibodyDataset` + `collate_fn` — full DataLoader loop

All cells use synthetic data — no real OAS download needed.

In [ ]:
import sys, pathlib, tempfile, numpy as np
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
from torch.utils.data import DataLoader

## 1. OASFilter — sequence quality gating

In [ ]:
from src.data.filter import OASFilter

filt = OASFilter(
    min_heavy_len=60,
    max_heavy_len=200,
    min_light_len=60,
    max_light_len=200,
    max_cdr_h3_len=30,
)

good_row = {
    "productive_H": True,
    "vj_in_frame_H": True,
    "productive_L": True,
    "vj_in_frame_L": True,
    "sequence_alignment_aa_H": "E" * 120,   # valid heavy
    "sequence_alignment_aa_L": "D" * 110,   # valid light
    "cdr3_aa_H": "ARDYYYGMDV",               # short CDR-H3
}

bad_row = {
    "productive_H": False,  # rejected
    "vj_in_frame_H": True,
    "productive_L": True,
    "vj_in_frame_L": True,
    "sequence_alignment_aa_H": "E" * 120,
    "sequence_alignment_aa_L": "D" * 110,
    "cdr3_aa_H": "ARDYYYGMDV",
}

print(f"good_row passes: {filt(good_row)}")
print(f"bad_row  passes: {filt(bad_row)}")

## 2. CDRSpanExtractor — IMGT numbering (synthetic spans)

In [ ]:
import torch

# Without ANARCI installed we demonstrate the span tensor format directly.
# cdr_spans: Tensor[6, 2]  = (start, end) for H1 H2 H3 L1 L2 L3
# Indices into the concatenated struct_emb (heavy then light).

N_heavy = 120
N_light = 110
N = N_heavy + N_light

# Synthetic CDR spans (IMGT approx positions)
cdr_spans = torch.tensor([
    [26, 32],          # H-CDR1  (IMGT 27-38  → 0-indexed 26-37)
    [50, 57],          # H-CDR2
    [95, 102],         # H-CDR3
    [N_heavy + 23, N_heavy + 33],  # L-CDR1  (offset by heavy length)
    [N_heavy + 49, N_heavy + 53],  # L-CDR2
    [N_heavy + 88, N_heavy + 96],  # L-CDR3
], dtype=torch.long)

print("cdr_spans shape:", cdr_spans.shape)
print("cdr_spans:")
labels = ["H-CDR1", "H-CDR2", "H-CDR3", "L-CDR1", "L-CDR2", "L-CDR3"]
for label, span in zip(labels, cdr_spans):
    print(f"  {label}: [{span[0].item()}, {span[1].item()})  len={span[1]-span[0]}")

## 3. EmbeddingCache — save and load structure embeddings

In [ ]:
from src.data.embeddings import EmbeddingCache

with tempfile.TemporaryDirectory() as tmpdir:
    cache = EmbeddingCache(tmpdir)

    # Write synthetic embeddings for two antibodies
    for ab_id in ["AB001", "AB002"]:
        emb   = np.random.randn(230, 512).astype(np.float32)  # (N, d_e)
        spans = np.array([[26,32],[50,57],[95,102],[143,153],[169,173],[208,216]], dtype=np.int64)
        mask  = np.ones(230, dtype=bool)
        plddt = np.random.uniform(70, 100, size=230).astype(np.float32)
        cache.save(ab_id, emb, spans, mask, plddt)

    # Load back
    data = cache.load("AB001")
    print("Keys:", list(data.keys()))
    print(f"struct_emb : {data['struct_emb'].shape}  dtype={data['struct_emb'].dtype}")
    print(f"cdr_spans  : {data['cdr_spans'].shape}")
    print(f"pad_mask   : {data['pad_mask'].shape}")
    print(f"plddt      : {data['plddt'].shape}  mean={data['plddt'].mean():.1f}")

## 4. AntibodyDataset — full __getitem__ contract

In [ ]:
from src.data.dataset import AntibodyDataset, collate_fn
from src.utils.tokenizer import AntibodyTokenizer
import pandas as pd

tok = AntibodyTokenizer()

# Build a tiny synthetic dataset on disk
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = pathlib.Path(tmpdir)
    cache_dir = tmpdir / "embeddings"
    cache_dir.mkdir()

    rows = []
    cache = EmbeddingCache(str(cache_dir))
    for i in range(4):
        ab_id   = f"AB{i:03d}"
        N_h, N_l = np.random.randint(100, 130), np.random.randint(95, 120)
        N       = N_h + N_l
        heavy   = "".join(np.random.choice(list("ACDEFGHIKLMNPQRSTVWY"), N_h))
        light   = "".join(np.random.choice(list("ACDEFGHIKLMNPQRSTVWY"), N_l))
        emb     = np.random.randn(N, 512).astype(np.float32)
        spans   = np.array([[26,32],[50,57],[95,102],[N_h+23,N_h+33],[N_h+49,N_h+53],[N_h+88,N_h+96]], dtype=np.int64)
        spans   = np.clip(spans, 0, N - 1)
        mask    = np.ones(N, dtype=bool)
        plddt   = np.random.uniform(70, 100, size=N).astype(np.float32)
        cache.save(ab_id, emb, spans, mask, plddt)
        rows.append({"id": ab_id, "heavy_seq": heavy, "light_seq": light})

    df = pd.DataFrame(rows)
    parquet_path = tmpdir / "data.parquet"
    df.to_parquet(parquet_path, index=False)

    ds = AntibodyDataset(
        parquet_path=str(parquet_path),
        cache_dir=str(cache_dir),
        tokenizer=tok,
    )
    print(f"Dataset length: {len(ds)}")
    item = ds[0]
    print("\nSingle item keys & shapes:")
    for k, v in item.items():
        if hasattr(v, 'shape'):
            print(f"  {k:12s}: {v.shape}  dtype={v.dtype}")
        else:
            print(f"  {k:12s}: {repr(v)[:60]}")

## 5. DataLoader with collate_fn — batch padding

In [ ]:
    loader = DataLoader(ds, batch_size=2, shuffle=False, collate_fn=collate_fn)
    batch = next(iter(loader))

    print("Batch keys & shapes after collation:")
    for k, v in batch.items():
        if hasattr(v, 'shape'):
            print(f"  {k:14s}: {v.shape}  dtype={v.dtype}")
        else:
            print(f"  {k:14s}: {v}")

    print("\nseq_pad_mask (True = real token):")
    print(batch['seq_pad_mask'])

---
**Summary**: The data pipeline goes CSV/Parquet → filter → embed (offline) → cache → `AntibodyDataset` → `collate_fn` → `DataLoader`. All shapes match the CONTRACT spec.